# Pre-Processing

## 1) Libraries and Data Loading

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Loading the dataset
df = pd.read_csv('cardio_train.csv', sep=';')

# Basic check of the data
print("Initial Shape:", df.shape)
df.head()

Initial Shape: (70000, 13)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


## 2) Dropping Irrelevant Features

### Drop Unnessary Column
* The `id` column is just an index and provides no predictive power.

In [2]:
# The 'id' column has no predictive value for medical outcomes
df.drop("id", axis=1, inplace=True)

print("Columns remaining:", df.columns.tolist())

Columns remaining: ['age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']


### 3: Age Transformation and Feature Renaming

In [3]:
# Converting age from days to years
df["age"] = (df["age"] / 365.25).round().astype(int)

# Renaming for better readability
df = df.rename(columns={
    'ap_hi': 'high_bp',
    'ap_lo': 'low_bp',
    'cholesterol': 'cholesterol',
    'gluc': 'glucose'
})

df.head()

,age,gender,height,weight,high_bp,low_bp,cholesterol,glucose,smoke,alco,active,cardio
0,50,2,168,62.0,110,80,1,1,0,0,1,0
1,55,1,156,85.0,140,90,3,1,0,0,1,1
2,52,1,165,64.0,130,70,3,1,0,0,0,1
3,48,2,169,82.0,150,100,1,1,0,0,1,1
4,48,1,156,56.0,100,60,1,1,0,0,0,0


### 4: Handling Blood Pressure Anomalies

In [4]:
# 1. Systolic (high_bp) must be greater than Diastolic (low_bp)
df = df[df['high_bp'] >= df['low_bp']]

# 2. Keep only medically plausible ranges
# Systolic: 70 to 250 | Diastolic: 40 to 150
df = df[(df["high_bp"] >= 70) & (df["high_bp"] <= 250)]
df = df[(df["low_bp"] >= 40) & (df["low_bp"] <= 150)]

print("Shape after BP cleaning:", df.shape)

Shape after BP cleaning: (68668, 12)


## 5: Outlier Capping (Height and Weight)

In [5]:
# Instead of deleting, we cap extreme values to avoid losing data
df['height'] = df['height'].clip(lower=120, upper=220)
df['weight'] = df['weight'].clip(lower=30, upper=200)

print("Height/Weight capping complete.")

Height/Weight capping complete.


## 6: Feature Engineering (BMI & Pulse Pressure)

In [6]:
# Calculate BMI: Weight(kg) / [Height(m)]^2
df['BMI'] = df["weight"] / ((df["height"] / 100) ** 2)

# Cap BMI to a realistic range (13 to 55)
df['BMI'] = df['BMI'].clip(lower=13, upper=55)

# Calculate Pulse Pressure (The difference between systolic and diastolic)
df['pulse_pressure'] = df['high_bp'] - df['low_bp']

df[['height', 'weight', 'BMI', 'pulse_pressure']].head()

,height,weight,BMI,pulse_pressure
0,168,62.0,21.967120,30
1,156,85.0,34.927679,50
2,165,64.0,23.507805,60
3,169,82.0,28.710479,50
4,156,56.0,23.011177,40


## 7: Categorical Encoding

In [ ]:
# Convert Cholesterol and Glucose into dummy variables
df = pd.get_dummies(
    df, 
    columns=['cholesterol', 'glucose'], 
    prefix=['chol', 'gluc'], 
    dtype=int
)

# Binary encoding for Gender (0: Female, 1: Male)
df['gender'] = df['gender'].map({1: 0, 2: 1})

df.head()

,age,gender,height,weight,high_bp,low_bp,smoke,alco,active,cardio,BMI,pulse_pressure,chol_1,chol_2,chol_3,gluc_1,gluc_2,gluc_3
0,50,1,168,62.0,110,80,0,0,1,0,21.967120,30,1,0,0,1,0,0
1,55,0,156,85.0,140,90,0,0,1,1,34.927679,50,0,0,1,1,0,0
2,52,0,165,64.0,130,70,0,0,0,1,23.507805,60,0,0,1,1,0,0
3,48,1,169,82.0,150,100,0,0,1,1,28.710479,50,1,0,0,1,0,0
4,48,0,156,56.0,100,60,0,0,0,0,23.011177,40,1,0,0,1,0,0


## 8: Final Feature Selection
#### We use BMI for training, so we drop the original height and weight to simplify the model and reduce redundant information

In [8]:
# Dropping original height/weight since we now use BMI
df.drop(columns=['height', 'weight'], inplace=True)

# Final duplicate removal after processing
df.drop_duplicates(inplace=True)

print("Final Pre-processed Shape:", df.shape)

Final Pre-processed Shape: (64838, 16)


## Finally Drop All Duplicates

## Final Database Varification

In [9]:
print(f"Rows and Columns after Pre Processing : {len(df)}, {df.shape[1]}")

Rows and Columns after Pre Processing : 64838, 16


In [10]:
df.info()

<class 'pandas.DataFrame'>
Index: 64838 entries, 0 to 69999
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             64838 non-null  int64  
 1   gender          64838 non-null  int64  
 2   high_bp         64838 non-null  int64  
 3   low_bp          64838 non-null  int64  
 4   smoke           64838 non-null  int64  
 5   alco            64838 non-null  int64  
 6   active          64838 non-null  int64  
 7   cardio          64838 non-null  int64  
 8   BMI             64838 non-null  float64
 9   pulse_pressure  64838 non-null  int64  
 10  chol_1          64838 non-null  int64  
 11  chol_2          64838 non-null  int64  
 12  chol_3          64838 non-null  int64  
 13  gluc_1          64838 non-null  int64  
 14  gluc_2          64838 non-null  int64  
 15  gluc_3          64838 non-null  int64  
dtypes: float64(1), int64(15)
memory usage: 8.4 MB


## 9: Data Export

In [11]:
# Save the cleaned dataset for the modeling task
df.to_csv("cardio_cleaned.csv", index=False)
print("File 'cardio_cleaned.csv' saved successfully!")

File 'cardio_cleaned.csv' saved successfully!


### **Task-2: Data Preprocessing Summary**

| Step | Action Taken | Logic / Threshold | Rationale |
| :--- | :--- | :--- | :--- |
| **ID Removal** | Dropped `id` column | Removed `df['id']` | The ID is a unique index and provides no predictive value for medical outcomes. |
| **Age Conversion** | Days to Years | `age / 365.25` | Converted to years to make the data more interpretable and reduce noise. |
| **Renaming** | Feature Clarity | `ap_hi` → `high_bp`, `ap_lo` → `low_bp` | Improved readability for medical features. |
| **BP Cleaning** | Logical Filtering | `high_bp >= low_bp` | Removed records where systolic pressure was lower than diastolic (physiologically impossible). |
| **BP Ranges** | Range Filtering | High: 70-250 \| Low: 40-150 | Removed extreme outliers and errors outside medically plausible ranges. |
| **Capping** | Outlier Handling | Height: 120-220 \| Weight: 30-200 | Replaced extreme outliers with boundary values to retain data points without skewing the model. |
| **New Feature** | BMI Calculation | `weight / (height/100)^2` | Combined height and weight into a single Body Mass Index metric, which is a better heart disease indicator. |
| **New Feature** | Pulse Pressure | `high_bp - low_bp` | Created a new clinical metric representing the force that the heart generates each time it contracts. |
| **Encoding** | One-Hot Encoding | Cholesterol & Glucose | Converted multi-level categorical features into binary columns for machine learning compatibility. |
| **Mapping** | Binary Encoding | Gender (1=0, 2=1) | Standardized gender into a 0/1 binary format. |
| **Selection** | Feature Dropping | Dropped `height` & `weight` | Removed original metrics after calculating BMI to reduce redundancy (multicollinearity). |
| **Final Clean** | Duplicate Removal | `df.drop_duplicates()` | Removed identical records created after reducing data granularity (like age conversion). |